# Lesson 28 — Bilevel and Complementarity Programs

## Learning objectives

1. Write a bilevel optimization problem and recognize its structure.
2. Reformulate a convex-lower-level bilevel as MPCC / MPEC via KKT.
3. Use big-M relaxations of complementarity constraints.
4. Recognize Stackelberg games and toll-setting as bilevel.
5. Use `discopt`'s native complementarity (`m.complementarity`, `discopt.mpec`) and bilevel (`discopt.bilevel.BilevelProblem`) tools.

## 1. Bilevel structure

$$
\begin{aligned}
\min_{x, y}\quad & F(x, y) \\
\text{s.t.}\quad & G(x, y) \le 0 \\
& y \in \arg\min_{y'} \{ f(x, y') : g(x, y') \le 0 \}.
\end{aligned}
$$

Outer problem (leader): $x$. Inner problem (follower): $y$ depends on $x$. Examples: pricing where competitors react, defender-attacker games, network design with user-equilibrium routing {cite:p}`Dempe2002,LuoPangRalph1996`.

Bilevel is *strictly harder* than the union of two NLPs — even with linear inner and outer it is NP-hard.

## 2. KKT reformulation

If the inner problem is convex with strong duality (Slater holds), replace $y \in \arg\min$ with the inner KKT conditions. With $g : \mathbb{R}^{n+m} \to \mathbb{R}^p$ and Jacobian $\nabla_y g(x, y) \in \mathbb{R}^{p \times m}$:

$$\nabla_y f(x, y) + \nabla_y g(x, y)^\top \lambda = 0, \quad g(x, y) \le 0, \quad \lambda \ge 0, \quad \lambda \circ g(x, y) = 0.$$

The complementarity $\lambda \circ g = 0$ is the troublesome bit — it's nonconvex (a union of $2^p$ faces depending on which inequalities are active). The reformulated problem is a **Mathematical Program with Complementarity Constraints (MPCC)** {cite:p}`LuoPangRalph1996`.

## 3. Solving MPCC

Three approaches:

- **Big-M / disjunctive MILP**: introduce binary $z_i$, add $\lambda_i \le M z_i$, $-g_i \le M(1 - z_i)$. Tight bounds critical.
- **Penalty / regularization**: $\lambda^\top g \le \epsilon$ for shrinking $\epsilon$.
- **Active-set**: enumerate $\lambda_i > 0$ vs $g_i < 0$ partitions.

Specialized solvers: PATH, MILES (mixed complementarity); IPOPT with NLP-relaxation often works for small MPCC. The penalty/regularization route is Scholtes' scheme {cite:p}`Scholtes2001`: replace $\lambda_i g_i = 0$ by $\lambda_i (-g_i) \le t$ and drive $t \to 0$ through a sequence of local NLP solves — fast, but only locally optimal.

## 4. Stackelberg pricing

Leader sets prices $p$; follower (consumer) chooses quantity $q(p)$ minimizing cost. Leader maximizes revenue $p \cdot q(p)$. The follower's best-response problem is convex; its KKT becomes the bilevel constraint.

Big-M reformulation gives a MILP for piecewise-linear demand.

## 5. Network toll-setting

You set tolls on a road network; users follow Wardrop equilibrium (each driver picks shortest-cost path). Maximize revenue subject to user equilibrium.

User-equilibrium → variational inequality → KKT → MPCC. Solved at scale only for moderate networks; instances with $n \sim 10^3$ links are challenging.

## References
{cite:p}`Dempe2002,LuoPangRalph1996`.

In [1]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [2]:
import numpy as np, discopt as do
from discopt.modeling import Model

# Toy bilevel: leader picks tax t in [0, 5]; follower picks q in [0, 10]
# minimizing (q - 4)^2 + t * q.
# Leader maximizes revenue t * q*(t).
# Inner KKT: 2(q - 4) + t - lam = 0,  lam * q = 0,  lam, q >= 0.
# So q* = max(0, 4 - t/2). Leader: max_t t * (4 - t/2). Optimum t* = 4,
# q* = 2, revenue = 8.
#
# The complementarity lam * q = 0 is a bilinear (non-convex) equality, and a
# bilinear equality violates the usual constraint qualifications. Here we
# write the standard big-M disjunctive reformulation by hand:
#     y = 1  -> lam = 0           (`lam <= M (1 - y)`)
#     y = 0  -> q   = 0           (`q   <= M y`)
# which leaves a MINLP whose only nonconvexity is the bilinear objective.
m = Model("bilevel_toy")
t   = m.continuous("t",   lb=0, ub=5)
q   = m.continuous("q",   lb=0, ub=10)
lam = m.continuous("lam", lb=0, ub=20)         # KKT multiplier for q >= 0
y   = m.binary("y")                            # complementarity selector
M   = 20.0

m.subject_to(2*(q - 4) + t - lam == 0)         # inner stationarity
m.subject_to(lam <= M * (1 - y))               # y = 1  =>  lam = 0
m.subject_to(q   <= M * y)                     # y = 0  =>  q   = 0
m.maximize(t * q)                              # leader objective (bilinear)
r = m.solve(time_limit=30)
print("leader's tax:", round(float(r.value(t)), 4),
      " follower q:", round(float(r.value(q)), 4),
      " revenue:",    round(float(r.objective), 4))


leader's tax: 4.0  follower q: 2.0  revenue: 8.0


## 6. Native tools in `discopt`

You rarely need to write the big-M by hand:

- `m.complementarity(lam, q)` declares $0 \le \lambda \perp q \ge 0$ as a
  first-class relation. The default `method="gdp"` lowers it to a
  disjunction ($\lambda = 0$ or $q = 0$) with bounds-derived big-M, so the
  global solve certifies the result; `"sos1"` uses an SOS1 set instead.
  `m.mcp(F, z, lb, ub)` states a box-constrained mixed complementarity
  problem.
- `discopt.mpec.solve_mpec(m, pairs, method="scholtes" | "sos1" | "gdp")`
  runs the chosen scheme and attaches `r.mpec_report` (complementarity and
  feasibility residuals measured on the *source* relation). Scholtes is a
  local homotopy and reports `"local_optimal"` with no bound.
- `discopt.bilevel.BilevelProblem(m, upper_vars, lower_vars,
  lower_objective, lower_constraints)` writes the follower's KKT system for
  you (symbolic differentiation, convexity gate on the lower level) and
  `formulate(method="kkt", mpec_method="gdp" | "sos1")` or
  `formulate(method="strong_duality")` turns it into a single-level model.
  The big-M encodings need an asserted bound on the follower multipliers,
  `multiplier_ub=...`, and refuse to guess one.

In [3]:
from discopt.bilevel import BilevelProblem
from discopt.mpec import complementarity, solve_mpec

def leader():
    mm = Model("bilevel_toy_native")
    tt = mm.continuous("t", lb=0, ub=5)
    qq = mm.continuous("q", lb=0, ub=10)
    mm.maximize(tt * qq)
    return mm, tt, qq

# (a) hand-written KKT stationarity + native complementarity
mm, tt, qq = leader()
ll = mm.continuous("lam", lb=0, ub=20)
mm.subject_to(2 * (qq - 4) + tt - ll == 0)
mm.complementarity(ll, qq)                      # 0 <= lam  _|_  q >= 0
ra = mm.solve(time_limit=30)
print(f"m.complementarity     : {ra.status:13s} t={float(ra.value(tt)):.4f} q={float(ra.value(qq)):.4f} revenue={ra.objective:.4f}")

# (b) the whole reduction from the follower's problem
for method, kw in [("strong_duality", {}), ("kkt", {"multiplier_ub": 50.0})]:
    mm, tt, qq = leader()
    bl = BilevelProblem(mm, upper_vars=[tt], lower_vars=[qq],
                        lower_objective=(qq - 4)**2 + tt * qq,
                        lower_constraints=[], **kw)     # q's bounds are folded in
    bl.formulate(method=method)
    rb = mm.solve(time_limit=30)
    print(f"BilevelProblem {method:14s}: {rb.status:13s} t={float(rb.value(tt)):.4f} q={float(rb.value(qq)):.4f} "
          f"revenue={rb.objective:.4f} certified={rb.gap_certified}")

# (c) Scholtes regularization: a local method, so no certificate
mm, tt, qq = leader()
ll = mm.continuous("lam", lb=0, ub=20)
mm.subject_to(2 * (qq - 4) + tt - ll == 0)
rc = solve_mpec(mm, [complementarity(ll, qq)], method="scholtes")
print(f"Scholtes              : {rc.status:13s} revenue={rc.objective:.4f} bound={rc.bound}")


m.complementarity     : optimal       t=4.0000 q=2.0000 revenue=8.0000
BilevelProblem strong_duality: optimal       t=4.0000 q=2.0000 revenue=8.0000 certified=True
BilevelProblem kkt           : optimal       t=4.0000 q=2.0000 revenue=8.0000 certified=True
Scholtes              : local_optimal revenue=8.0000 bound=None
